In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import pandas as pd

In [3]:
cadec_dir = "/content/drive/MyDrive/Pharmacovigilance/CADEC"

In [4]:
ann_folder = os.path.join(cadec_dir, "original")

In [5]:
label_map = {
    "Drug": "DRUG",
    "ADR": "EFFECT"
}

In [6]:
rows = []

In [7]:
ann_files = sorted(os.listdir(ann_folder))

In [ ]:
for file in ann_files:

    if not file.endswith(".ann"):
        continue

    name = file.replace(".ann", "")

    text_path = os.path.join(cadec_dir, "text", name + ".txt")

    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()

    spans = []
    other_spans = []

    ann_path = os.path.join(ann_folder, file)

    with open(ann_path, "r", encoding="utf-8") as f:

        for line in f:

            if not line.startswith("T"):
                continue

            parts = line.strip().split("\t")

            if len(parts) < 3:
                continue

            annotation = parts[1].split(" ", 1)

            label = annotation[0]
            offsets = annotation[1]

            for piece in offsets.split(";"):

                start, end = piece.split()

                start = int(start)
                end = int(end)

                if label in label_map:

                    spans.append({
                        "start": start,
                        "end": end,
                        "label": label_map[label]
                    })

                else:

                    other_spans.append({
                        "start": start,
                        "end": end,
                        "label": label
                    })

    rows.append({
        "post": name,
        "text": text,
        "spans": spans,
        "other_spans": other_spans
    })


In [ ]:
cadec_df = pd.DataFrame(rows)

print("Total posts:", len(cadec_df))



In [ ]:
cadec_df

In [ ]:

for index, row in cadec_df.head(5).iterrows():

    print("\nPOST:", row["post"])
    print("TEXT:", row["text"][:300])

    print("\nTarget annotations:")

    for span in row["spans"]:

        extracted = row["text"][span["start"]:span["end"]]

        print(span["label"], "->", extracted)

    print("\nOther annotations:")

    for span in row["other_spans"]:

        extracted = row["text"][span["start"]:span["end"]]

        print(span["label"], "->", extracted)

    print("-" * 60)


In [ ]:
labels = [
    "O",
    "B-DRUG",
    "I-DRUG",
    "B-EFFECT",
    "I-EFFECT"
]

label2id = {}

for index, label in enumerate(labels):
    label2id[label] = index

id2label = {}

for index, label in enumerate(labels):
    id2label[index] = label

print(label2id)